# Notebook 12 — Local ViT-B/16 checkpoint restoration

Load the final `vit_pretrained_best_checkpoint.pt` with the exact Residual Attention Pooling architecture from the final Kaggle ensemble notebook, then run a one-valid-slice software smoke test.

**Sources:** `08_model_ensemble_kaggle_dataset_ready.ipynb`, Cell 17; the checkpoint's saved `run_config` is authoritative for the model dimensions and center-prior parameters.

This notebook does not yet run all 36 slices or produce a final ViT study prediction.

In [1]:
from pathlib import Path
import sys, json, time, gc
import numpy as np
import pandas as pd
import torch

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "rsna_knee").is_dir():
            return candidate
    raise FileNotFoundError("Could not find project root containing src/rsna_knee")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from rsna_knee.models.vit_study import TARGETS
from rsna_knee.models.vit_checkpoint import resolve_vit_artifact, load_vit_model

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Smaller chunks reduce peak memory on CPU; this is a runtime-only setting.
RUNTIME_SLICE_CHUNK_SIZE = 2 if DEVICE.type == "cpu" else 8

print("Project root:", PROJECT_ROOT)
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
print("Runtime slice chunk:", RUNTIME_SLICE_CHUNK_SIZE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Project root: C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection
Python: 3.13.9
PyTorch: 2.11.0+cpu
Device: cpu
Runtime slice chunk: 2


In [2]:
PREPROCESSED_PATH = (
    PROJECT_ROOT / "demo" / "output" / "vit_preprocessing"
    / "vit_preprocessed_sample.npz"
)
if not PREPROCESSED_PATH.is_file():
    raise FileNotFoundError(
        f"ViT Phase 2D preprocessing cache not found: {PREPROCESSED_PATH}. "
        "Run notebooks/11_local_vit_preprocessing.ipynb first."
    )

with np.load(PREPROCESSED_PATH, allow_pickle=False) as data:
    images_np = data["images"].copy()
    valid_np = data["valid"].copy()
    plane_ids_np = data["plane_ids"].copy()
    fluid_np = data["fluid"].copy()
    fat_np = data["fat"].copy()
    positions_np = data["positions"].copy()
    saved_plane_order = data["plane_order"].astype(str).tolist()
    series_uids = data["series_uids"].astype(str).tolist()
    study_uid = str(data["study_uid"].item())

assert saved_plane_order == ["Sagittal", "Coronal", "Axial"], saved_plane_order
assert images_np.shape == (36, 224, 224), images_np.shape
assert images_np.dtype == np.uint8, images_np.dtype
for name, arr in {
    "valid": valid_np, "plane_ids": plane_ids_np, "fluid": fluid_np,
    "fat": fat_np, "positions": positions_np,
}.items():
    assert arr.shape == (36,), (name, arr.shape)
assert int(valid_np.sum()) > 0
assert np.isfinite(positions_np).all()

print("Study UID:", study_uid)
print("Images:", images_np.shape, images_np.dtype)
print("Valid slices:", int(valid_np.sum()), "/ 36")
print("Plane counts:", {int(k): int(v) for k, v in zip(*np.unique(plane_ids_np[valid_np.astype(bool)], return_counts=True))})
print("Series UIDs:", dict(zip(saved_plane_order, series_uids)))
print("[PASS] Phase 2D preprocessing cache contract")

Study UID: 1.2.826.0.1.3680043.8.498.10095687747295410396510538520594649149
Images: (36, 224, 224) uint8
Valid slices: 36 / 36
Plane counts: {0: 12, 1: 12, 2: 12}
Series UIDs: {'Sagittal': '1.2.826.0.1.3680043.8.498.37372672584387115708078590109474719429', 'Coronal': '1.2.826.0.1.3680043.8.498.21497837050858589852101556928971904198', 'Axial': '1.2.826.0.1.3680043.8.498.67495435033389756963800730529101229430'}
[PASS] Phase 2D preprocessing cache contract


In [3]:
# The known final artifact from the user's project. Set this to another path if needed.
configured_artifact = (
    PROJECT_ROOT / "artifacts" / "checkpoints" / "102526"
    / "vit_pretrained_centerprior_model_artifacts.zip"
)
VIT_ARTIFACT_SOURCE = configured_artifact if configured_artifact.exists() else None
vit_artifact = resolve_vit_artifact(VIT_ARTIFACT_SOURCE, PROJECT_ROOT)

print("ViT artifact:", vit_artifact)
print("Artifact size (GiB):", round(vit_artifact.stat().st_size / 1024**3, 3))

ViT artifact: C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection\artifacts\checkpoints\102526\vit_pretrained_centerprior_model_artifacts.zip
Artifact size (GiB): 0.728


In [4]:
started = time.time()
model, model_info = load_vit_model(
    vit_artifact,
    device=DEVICE,
    runtime_slice_chunk_size=RUNTIME_SLICE_CHUNK_SIZE,
)
restore_seconds = time.time() - started

print("========== VIT CHECKPOINT RESTORE ==========")
print("Architecture:", model_info["architecture"])
print("Best epoch:", model_info["best_epoch"])
print("Best validation BCE:", model_info["best_val_loss"])
print("Strict state-dict load:", model_info["strict_load"])
print("Parameter count (M):", round(model_info["parameter_count"] / 1e6, 3))
print("Device:", model_info["device"])
print("Restore seconds:", round(restore_seconds, 2))
print("Runtime slice chunk:", model.config.slice_chunk_size)
assert model_info["strict_load"] is True

========== VIT CHECKPOINT RESTORE ==========
Architecture: Pretrained ViT-B/16 + gated metadata + study Transformer + residual mean-attention pooling
Best epoch: 7
Best validation BCE: 0.6440770427385966
Strict state-dict load: True
Parameter count (M): 88.788
Device: cpu
Restore seconds: 21.17
Runtime slice chunk: 2


## 5. Quick forward smoke test

Giữ tensor đầy đủ `[1, 36, 224, 224]` và metadata tương ứng, nhưng chỉ đánh dấu một slice hợp lệ để kiểm tra model graph, normalization, torchvision ViT-B/16, study aggregator và 12 output mà không chạy encoder trên toàn bộ study.

Đây là software smoke test, không phải prediction cuối cùng, đánh giá y khoa hay competition score.

In [5]:
valid_indices = np.flatnonzero(valid_np.astype(bool))
if len(valid_indices) == 0:
    raise RuntimeError("Preprocessed sample has no valid slices.")
selected_index = int(valid_indices[len(valid_indices) // 2])

quick_valid = np.zeros((1, 36), dtype=np.bool_)
quick_valid[0, selected_index] = True

images_t = torch.from_numpy(images_np[None]).to(DEVICE)
valid_t = torch.from_numpy(quick_valid).to(DEVICE)
plane_t = torch.from_numpy(plane_ids_np.astype(np.int64)[None]).to(DEVICE)
fluid_t = torch.from_numpy(fluid_np.astype(np.int64)[None]).to(DEVICE)
fat_t = torch.from_numpy(fat_np.astype(np.int64)[None]).to(DEVICE)
positions_t = torch.from_numpy(positions_np.astype(np.float32)[None]).to(DEVICE)

with torch.inference_mode():
    logits = model(images_t, valid_t, plane_t, fluid_t, fat_t, positions_t)
    probabilities = torch.sigmoid(logits.float())

logits_np = logits.detach().cpu().numpy()[0]
probabilities_np = probabilities.detach().cpu().numpy()[0]
assert logits_np.shape == (12,), logits_np.shape
assert probabilities_np.shape == (12,), probabilities_np.shape
assert np.isfinite(logits_np).all()
assert np.isfinite(probabilities_np).all()
assert ((probabilities_np >= 0) & (probabilities_np <= 1)).all()

print("Selected valid slice index:", selected_index)
print("Forward logits shape:", tuple(logits.shape))
print("Probability range:", float(probabilities_np.min()), "to", float(probabilities_np.max()))
print("[PASS] one-valid-slice forward smoke test")

Selected valid slice index: 18
Forward logits shape: (1, 12)
Probability range: 0.19496490061283112 to 0.43098434805870056
[PASS] one-valid-slice forward smoke test


In [6]:
OUTPUT_DIR = PROJECT_ROOT / "demo" / "output" / "vit_model_restore"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
report_path = OUTPUT_DIR / "vit_checkpoint_restore_report.json"

report = {
    "status": "success",
    "study_instance_uid": study_uid,
    "artifact": model_info["artifact"],
    "artifact_member": model_info.get("member"),
    "checkpoint_name": model_info["checkpoint_name"],
    "architecture": model_info["architecture"],
    "best_epoch": model_info["best_epoch"],
    "best_val_loss": model_info["best_val_loss"],
    "strict_load": model_info["strict_load"],
    "parameter_count": model_info["parameter_count"],
    "device": model_info["device"],
    "runtime_config": model_info["run_config"],
    "smoke_test": {
        "valid_slice_index": selected_index,
        "logits_shape": list(logits_np.shape),
        "probability_shape": list(probabilities_np.shape),
        "all_finite": bool(np.isfinite(probabilities_np).all()),
        "probability_min": float(probabilities_np.min()),
        "probability_max": float(probabilities_np.max()),
        "note": "One-valid-slice software smoke test only; not a final study prediction.",
    },
    "restore_seconds": restore_seconds,
}
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
pd.DataFrame({"target": TARGETS, "smoke_test_probability": probabilities_np}).to_csv(
    OUTPUT_DIR / "vit_smoke_test_probabilities.csv", index=False
)

print("Saved:", report_path)
print("Saved:", OUTPUT_DIR / "vit_smoke_test_probabilities.csv")

Saved: C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection\demo\output\vit_model_restore\vit_checkpoint_restore_report.json
Saved: C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection\demo\output\vit_model_restore\vit_smoke_test_probabilities.csv


## 6. CRITERIA — Phase 2D checkpoint restoration

- [ ] The Phase 2D NPZ contract is valid.
- [ ] The final `vit_pretrained_best_checkpoint.pt` is resolved from the final artifact ZIP or direct checkpoint.
- [ ] Checkpoint target order matches the 12 competition targets.
- [ ] Checkpoint config matches ViT-B/16, 224px, 36 slices, embed dim 256, study depth 2 and 4 heads.
- [ ] `model.load_state_dict(..., strict=True)` succeeds with no missing or unexpected keys.
- [ ] The one-valid-slice forward smoke test returns 12 finite logits/probabilities in `[0, 1]`.
- [ ] The restore report and smoke-test CSV are saved.

**Next phase:** full-study ViT inference over all valid slices, followed by the 90% DINOv2 + 10% ViT ensemble. Do not use the one-slice smoke-test probabilities as model predictions.